In [1]:
from dotenv import load_dotenv
load_dotenv()

import pandas as pd
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()

docs = pd.read_csv("travel_docs.csv")      # doc_id, category, source, title, body
print(docs.shape)

# 문서 번호 하나로 자료 가져오기
def get_doc(doc_id: str) -> str:
    r = docs[docs["doc_id"] == doc_id].iloc[0]
    return f"[{r['doc_id']}] {r['title']}\n{r['body']}"

# 문서 번호 여러 개를 한 번에
def get_docs(doc_ids: list) -> str:
    return "\n\n".join(get_doc(i) for i in doc_ids)

print(get_doc("T02"))

(22, 5)
[T02] 별빛전망대
별빛전망대는 솔바람섬 북쪽 언덕 꼭대기에 있는 천문 관측 명소이다. 맑은 날 밤에는 은하수를 맨눈으로 볼 수 있다. 별빛전망대는 매주 월요일에 휴관하며 입장료는 5천 원이다. 전망대까지는 섬 선착장에서 걸어서 약 30분이 걸린다. 겨울철에는 오후 9시에 문을 닫는다.


In [2]:
prompt = ChatPromptTemplate.from_template("당신은 은하군 여행 안내원입니다. {question}에 답해주세요.")
chain = prompt | llm | parser

print(chain.invoke({"question": "별빛전망대는 언제 쉬어?"}))

별빛전망대의 운영 시간은 계절이나 특별한 이벤트에 따라 다를 수 있습니다. 일반적으로 주말이나 공휴일에는 운영하지만, 특정 날짜에 휴무일이 있을 수 있습니다. 정확한 정보를 원하신다면 별빛전망대의 공식 웹사이트나 고객센터에 문의하시는 것이 좋습니다. 여행 계획에 도움이 필요하시면 언제든지 말씀해 주세요!


In [3]:
prompt = ChatPromptTemplate.from_template(
    "아래 [자료]에 있는 내용만으로 답하세요.\n\n[자료]\n{context}\n\n[질문]\n{question}"
)
rag_chain = prompt | llm | parser

print(rag_chain.invoke({"context": get_doc("T02"), "question": "별빛전망대는 언제 쉬어?"}))

별빛전망대는 매주 월요일에 휴관합니다.


In [4]:
prompt = ChatPromptTemplate.from_template(
    "아래 [자료]에 있는 내용만으로 답하세요. [자료]에 없으면 '자료에 없습니다'라고 답하세요.\n\n"
    "[자료]\n{context}\n\n[질문]\n{question}"
)
rag_chain = prompt | llm | parser

print(rag_chain.invoke({"context": get_doc("T02"), "question": "별빛전망대 근처 맛집 알려줘"}))

자료에 없습니다.


In [5]:
prompt = ChatPromptTemplate.from_template(
    "아래 [자료]에 있는 내용만으로 답하세요. [자료]에 없으면 '자료에 없습니다'라고 답하세요.\n\n"
    "[자료]\n{context}\n\n[질문]\n{question}"
)
rag_chain = prompt | llm | parser

# batch: 입력 여러 개를 리스트로 한 번에
inputs = [
    {"context": get_doc("T09"), "question": "갈매기식당은 언제 쉬어?"},
    {"context": get_doc("T11"), "question": "솔바람페리는 하루 몇 번 다녀?"},
    {"context": get_doc("T13"), "question": "반딧불 트레킹 가이드는 누구야?"},
]
for r in rag_chain.batch(inputs):
    print(r)

# stream: 답을 조금씩 받아서 바로 출력
for chunk in rag_chain.stream({"context": get_doc("T15"),
                               "question": "취소하면 환불을 얼마나 받아? 자세히 설명해줘"}):
    print(chunk, end="", flush=True)

갈매기식당은 매주 화요일에 쉰다.
솔바람페리는 하루 4회 운항합니다.
반딧불 트레킹의 담당 가이드는 윤솔입니다.
바람길여행 상품의 취소 및 환불 규정은 다음과 같습니다:

1. 출발 10일 전까지 취소: 전액 환불
2. 출발 5일에서 9일 전 사이에 취소: 80% 환불
3. 출발 2일에서 4일 전 사이에 취소: 30% 환불
4. 출발 1일 전부터는 환불되지 않음

또한, 숙소 비용은 해당 숙소 운영사의 취소 규정을 따르며, 환불은 결제 수단으로 영업일 기준 5일 안에 처리됩니다. 단체 예약은 별도의 계약 조건을 따릅니다.

In [7]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 은하군 여행 안내원입니다. 아래 [자료]에 있는 내용만으로 답하세요."
               "[자료]에 없으면 '자료에 없습니다'라고 답하세요."),
    ("human", "[자료]\n{context}\n\n[질문]\n{question}"),
])
rag_chain = prompt | llm | parser

# 모델에 무엇이 전달되는지 확인 (모델은 부르지 않음)
pv = prompt.invoke({"context": get_doc("T02"), "question": "별빛전망대는 언제 쉬어?"})
for m in pv.to_messages():
    print(type(m).__name__, ":", m.content[:40])

print(rag_chain.invoke({"context": get_doc("T02"), "question": "별빛전망대는 언제 쉬어?"}))

SystemMessage : 당신은 은하군 여행 안내원입니다. 아래 [자료]에 있는 내용만으로 답하세
HumanMessage : [자료]
[T02] 별빛전망대
별빛전망대는 솔바람섬 북쪽 언덕 꼭대기에 
별빛전망대는 매주 월요일에 휴관합니다.


In [8]:
role_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role}입니다. 아래 [자료]에 있는 내용만으로 답하세요."),
    ("human", "[자료]\n{context}\n\n[질문]\n{question}"),
])
role_chain = role_prompt | llm | parser

for role in ["친절한 여행 상담원", "10살 아이에게 설명하는 안내원"]:
    print(f"【{role}】")
    print(role_chain.invoke({"role": role, "context": get_doc("T12"), "question": "이 상품 소개해줘"}), "\n")

【친절한 여행 상담원】
솔바람섬 별빛 2박3일은 바람길여행에서 제공하는 매력적인 투어 상품입니다. 가격은 1인 기준으로 39만 원이며, VAT가 포함되어 있지만 유류할증료는 별도로 발생합니다. 

여행 일정에는 파도소리 리조트에서의 2박 숙박이 포함되어 있으며, 특별한 경험으로 별빛전망대에서의 야간 관측과 은빛해변 산책이 포함되어 있습니다. 또한, 솔바람페리 왕복 승선권도 제공되어 편리하게 이동할 수 있습니다. 

이 투어의 담당 가이드는 한별이며, 최소 출발 인원은 6명입니다. 참고로, 날씨가 흐릴 경우 관측 일정은 다음 날로 연기될 수 있습니다. 아름다운 자연과 특별한 경험을 원하신다면 이 상품을 고려해보세요! 

【10살 아이에게 설명하는 안내원】
솔바람섬 별빛 2박3일은 바람길여행에서 판매하는 특별한 여행 상품이에요. 이 여행은 1인당 39만 원이고, 숙소는 파도소리 리조트에서 2박을 하게 돼요. 여행 일정에는 별빛전망대에서 밤하늘의 별을 관측하는 시간과 은빛해변을 산책하는 시간이 포함되어 있어요. 또한, 솔바람페리 왕복 승선권도 포함되어 있어서 섬으로 가는 배를 탈 수 있어요. 이 여행을 안내해 줄 담당 가이드는 한별이라는 분이에요. 여행을 가려면 최소 6명이 모여야 하고, 만약 날씨가 흐리면 별 관측은 다음 날로 미뤄질 수 있어요. 정말 멋진 여행이 될 것 같아요! 



In [9]:
form_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 은하군 여행 안내원입니다. 아래 [자료]에 있는 내용만으로 답하세요.\n"
               "반드시 아래 형식으로만 답하세요.\n"
               "답: (한두 문장)\n"
               "주의: (예약·휴무 등 주의할 점. 없으면 '없음')"),
    ("human", "[자료]\n{context}\n\n[질문]\n{question}"),
])
tests = [
    {"context": get_doc("T07"), "question": "계곡쉼 펜션에 강아지 데려가도 돼?"},
    {"context": get_doc("T04"), "question": "반딧불 축제 야간 탐방 그냥 가면 돼?"},
    {"context": get_doc("T10"), "question": "숲속부엌 대표 메뉴가 뭐야?"},
]
for r in (form_prompt | llm | parser).batch(tests):
    print(r, "\n")

답: 네, 계곡쉼 펜션은 반려동물 동반 투숙이 가능합니다.  
주의: 주말에는 예약이 빨리 마감되니 미리 예약하세요. 

답: 반딧불 축제의 야간 탐방은 사전 예약제로 운영되므로 예약이 필요합니다.  
주의: 사전 예약이 필요하며, 손전등과 휴대폰 불빛 사용이 금지됩니다. 

답: 숲속부엌의 대표 메뉴는 산채비빔밥입니다.  
주의: 단체 손님은 하루 전에 예약해야 합니다. 



In [10]:
# 잘못된 예: 중괄호 한 번
wrong = ChatPromptTemplate.from_template(
    '형식: {"answer": "답", "found": true 또는 false}\n[자료]\n{context}\n[질문]\n{question}'
)
print(wrong.input_variables)

# 올바른 예: 중괄호 두 번
right = ChatPromptTemplate.from_template(
    '형식: {{"answer": "답", "found": true 또는 false}}\n[자료]\n{context}\n[질문]\n{question}'
)
print(right.input_variables)

['"answer"', 'context', 'question']
['context', 'question']


In [11]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 은하군 여행 안내원입니다. 아래 [자료]에 있는 내용만으로 답하세요. "
               "[자료]에 없으면 '자료에 없습니다'라고 답하세요."),
    ("human", "[자료]\n{context}\n\n[질문]\n{question}"),
])
rag_chain = prompt | llm | parser

data = {"context": get_doc("T15"), "question": "출발 3일 전에 취소하면?"}

msg = (prompt | llm).invoke(data)            # 파서 없음
print(type(msg).__name__, "|", msg.content)

text = (prompt | llm | parser).invoke(data)  # StrOutputParser
print(type(text).__name__, "|", text)

AIMessage | 출발 3일 전에 취소하면 30%를 환불받을 수 있습니다.
TextAccessor | 출발 3일 전에 취소하면 30%를 환불받을 수 있습니다.


In [12]:
from langchain_core.output_parsers import JsonOutputParser

extract = ChatPromptTemplate.from_template(
    "다음 상품 자료에서 정보를 뽑아 JSON으로만 답해. 다른 설명은 넣지 마.\n"
    '형식: {{"name": "상품명", "price": "가격", "guide": "가이드 이름"}}\n\n'
    "[자료]\n{context}"
)
extract_chain = extract | llm | JsonOutputParser()

rows = extract_chain.batch([{"context": get_doc(i)} for i in ["T12", "T13", "T14"]])
print(type(rows[0]))

table = pd.DataFrame(rows)
print(table)

# 표에서 바로 찾기: 가이드가 한별인 상품
print(table[table["guide"] == "한별"]["name"].tolist())

<class 'dict'>
           name    price guide
0  솔바람섬 별빛 2박3일    39만 원    한별
1  청연계곡 반딧불 트레킹  69,000원    윤솔
2     노을항 미식 투어    25만 원    한별
['솔바람섬 별빛 2박3일', '노을항 미식 투어']


In [13]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

kw_chain = ChatPromptTemplate.from_template(
    "다음 자료의 핵심 키워드 5개를 쉼표로 구분해서만 답해.\n\n{context}"
) | llm | CommaSeparatedListOutputParser()

ids = ["T01", "T02", "T03", "T04", "T05"]
results = kw_chain.batch([{"context": get_doc(i)} for i in ids])
for doc_id, kws in zip(ids, results):
    print(doc_id, kws)

T01 ['솔바람섬', '별빛전망대', '은빛해변', '파도소리 리조트', '솔바람페리']
T02 ['별빛전망대', '천문 관측', '은하수', '입장료', '휴관']
T03 ['청연계곡', '트레킹 코스', '반딧불숲', '물놀이', '가족 방문객']
T04 ['반딧불숲', '반딧불 축제', '청연계곡', '야간 탐방', '은하군청']
T05 ['노을항', '해산물 축제', '갈매기식당', '방파제', '카페']


In [14]:
from langchain_core.output_parsers import JsonOutputParser

prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 은하군 여행 안내원입니다. 아래 [자료]에 있는 내용만으로 답하세요. "
               "[자료]에 없으면 '자료에 없습니다'라고 답하세요."),
    ("human", "[자료]\n{context}\n\n[질문]\n{question}"),
])
rag_chain = prompt | llm | parser

judge = ChatPromptTemplate.from_template(
    "아래 [자료]로 [질문]에 답할 수 있는지 판정해 JSON으로만 답해. 다른 설명은 넣지 마.\n"
    '형식: {{"found": true 또는 false, "reason": "판단 이유 한 문장"}}\n\n'
    "[자료]\n{context}\n\n[질문]\n{question}"
)
judge_chain = judge | llm | JsonOutputParser()

tests = [("T09", "갈매기식당 쉬는 날이 언제야?"),
         ("T09", "갈매기식당에 주차장 있어?")]

for doc_id, q in tests:
    data = {"context": get_doc(doc_id), "question": q}
    verdict = judge_chain.invoke(data)
    print("【질문】", q)
    print("【판정】", verdict)
    if verdict["found"] == True:
        print("【답변】", rag_chain.invoke(data))
    else:
        print("【답변】 자료에 없는 내용입니다.")
    print()

【질문】 갈매기식당 쉬는 날이 언제야?
【판정】 {'found': True, 'reason': '자료에 갈매기식당이 매주 화요일에 쉰다고 명시되어 있기 때문'}
【답변】 갈매기식당은 매주 화요일에 쉰다.

【질문】 갈매기식당에 주차장 있어?
【판정】 {'found': False, 'reason': '자료에 주차장에 대한 정보가 포함되어 있지 않다.'}
【답변】 자료에 없는 내용입니다.



In [15]:
templates = [
    "다음 여행사 규정을 여행객이 이해하기 쉽게 풀어서 설명해줘. 숫자는 그대로 써줘:\n{text}",
    "다음 내용을 딱 3줄로 요약해줘. 환불 비율은 빠뜨리지 마:\n{text}",
    "다음 한국어를 외국인 여행객용 영어로 번역해줘:\n{text}",
]

result = get_doc("T15")                       # 첫 입력: 환불 규정 문서
for i, template in enumerate(templates, start=1):
    chain = ChatPromptTemplate.from_template(template) | llm | parser
    result = chain.invoke({"text": result})   # 앞 단계 결과가 다음 단계 입력
    print(f"【{i}단계】\n{result}\n")

【1단계】
바람길여행의 예약 취소와 환불 규정은 다음과 같습니다:

1. **출발 10일 전까지 취소**: 전액 환불이 가능합니다.
2. **출발 5일에서 9일 전 사이에 취소**: 결제한 금액의 80%가 환불됩니다.
3. **출발 2일에서 4일 전 사이에 취소**: 결제한 금액의 30%가 환불됩니다.
4. **출발 1일 전부터 취소**: 환불이 불가능합니다.

또한, 숙소 비용은 해당 숙소의 규정에 따라 다를 수 있습니다. 환불은 결제한 방법으로 영업일 기준 5일 이내에 처리됩니다. 단체 예약의 경우는 별도의 계약 조건이 적용됩니다.

【2단계】
바람길여행의 예약 취소 및 환불 규정은 다음과 같습니다: 출발 10일 전까지 취소 시 전액 환불, 5일에서 9일 전 사이 취소 시 80% 환불, 2일에서 4일 전 사이 취소 시 30% 환불, 1일 전부터는 환불 불가입니다. 숙소 비용은 별도의 규정이 적용되며, 환불은 결제 방법에 따라 영업일 기준 5일 이내에 처리됩니다.

【3단계】
Here are the cancellation and refund policies for Baramgil Travel: 

- Full refund for cancellations made 10 days before departure.
- 80% refund for cancellations made between 5 to 9 days before departure.
- 30% refund for cancellations made between 2 to 4 days before departure.
- No refund for cancellations made 1 day before departure or on the day of departure.

Please note that accommodation costs are subject to separate regulations, and refunds will be processed within 5 business days based on

In [16]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 은하군 여행 안내원입니다. 아래 [자료]에 있는 내용만으로 답하세요. "
               "[자료]에 없으면 '자료에 없습니다'라고 답하세요."),
    ("human", "[자료]\n{context}\n\n[질문]\n{question}"),
])
rag_chain = prompt | llm | parser

from langchain_core.runnables import RunnableLambda

# 입력 {"doc_ids": [...], "question": "..."}  →  프롬프트용 {"context": ..., "question": ...}
def to_prompt_input(x):
    return {"context": get_docs(x["doc_ids"]), "question": x["question"]}

doc_chain = RunnableLambda(to_prompt_input) | prompt | llm | parser

print(doc_chain.invoke({
    "doc_ids": ["T12", "T15", "T16"],
    "question": "솔바람섬 별빛 2박3일을 출발 5일 전에 취소하면 환불은 어떻게 돼?"
}))

솔바람섬 별빛 2박3일을 출발 5일 전에 취소하면 바람길여행의 규정에 따라 80% 환불됩니다. 단, 숙소 비용은 파도소리 리조트의 취소 규정에 따라 다를 수 있습니다. 파도소리 리조트의 규정에 따르면 체크인 3일에서 6일 전에 취소하면 50% 환불되므로, 숙소 비용은 50% 환불됩니다.


In [17]:
from langchain_core.runnables import RunnableLambda

def to_prompt_input(x):
    return {"context": get_docs(x["doc_ids"]), "question": x["question"]}

strict_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 은하군 여행 안내원입니다. 아래 [자료]에 있는 내용만으로 답하세요. "
               "[자료]에 없으면 '자료에 없습니다'라고 답하세요. "
               "질문에 나온 이름과 정확히 같은 이름의 자료만 근거로 쓰세요."),
    ("human", "[자료]\n{context}\n\n[질문]\n{question}"),
])
strict_chain = RunnableLambda(to_prompt_input) | strict_prompt | llm | parser

tests = [
    {"doc_ids": ["T15", "T22"], "question": "바람길여행 상품을 출발 3일 전에 취소하면 환불은?"},
    {"doc_ids": ["T06", "T20"], "question": "파도소리 리조트 운영사는?"},
]
for r in strict_chain.batch(tests):
    print(r)

바람길여행 상품은 출발 3일 전에 취소하면 30%를 환불받습니다.
파도소리 리조트의 운영사는 해온호스피탈리티입니다.


In [18]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 은하군 여행 안내원입니다. 아래 [자료]에 있는 내용만으로 답하세요. "
               "[자료]에 없으면 '자료에 없습니다'라고 답하세요."),
    ("human", "[자료]\n{context}\n\n[질문]\n{question}"),
])
rag_chain = prompt | llm | parser

import gradio as gr

choices = [f"{r.doc_id} {r.title}" for r in docs.itertuples()]   # 예: "T02 별빛전망대"

def process(choice, question):
    doc_id = choice.split()[0]          # "T02 별빛전망대" → "T02"
    context = get_doc(doc_id)
    answer = rag_chain.invoke({"context": context, "question": question})
    return context, answer

gr.Interface(
    fn=process,
    inputs=[gr.Dropdown(choices, label="문서 선택"), gr.Textbox(label="질문")],
    outputs=[gr.Textbox(lines=6, label="📄 모델에 전달된 자료"),
             gr.Textbox(lines=6, label="💬 답변")],
    title="은하군 여행 안내",
).launch()

c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
